In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
pd.options.display.float_format = '{:.2f}'.format
warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [4]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по верблюдам v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Верблюды
815,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2022-09-01,7.80
1051,МАНГИСТАУСКАЯ ОБЛАСТЬ,2021-05-01,134.30
1062,МАНГИСТАУСКАЯ ОБЛАСТЬ,2022-04-01,87.46
343,АТЫРАУСКАЯ ОБЛАСТЬ,2020-11-01,357.62
543,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2020-02-01,35.80
658,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2019-03-01,4.48
316,АТЫРАУСКАЯ ОБЛАСТЬ,2018-08-01,78.10
1042,МАНГИСТАУСКАЯ ОБЛАСТЬ,2020-08-01,152.90
497,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2016-04-01,22.13
765,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2017-09-01,6.22


In [5]:
regions = df['Регион'].unique()
target   = "Верблюды"
horizon  = 3
epsilon = 1e-6

In [6]:
first_test = pd.to_datetime("2024-08-01")
last_possible = df["Период"].max() - pd.DateOffset(months=horizon-1)
test_starts = pd.date_range(first_test, last_possible, freq="MS")

## Holw-Winter's (log)

In [7]:
results_hw = []
for region in regions:
    ts = (df[df["Регион"] == region]
          .set_index("Период")[target]
          .dropna()
          .sort_index())
    if len(ts) < 24:
        print(f"{region}: всего {len(ts)} мес. — сезонный Holt-Winter's невозможен.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        # формируем train / test
        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]

        # пропускаем, если недостаточно данных или неполный test
        if len(train) < 24 or len(test) < horizon:
            continue

        # обучаем модель
        train_log = np.log1p(train)

        hw_log = ExponentialSmoothing(
            train_log,
            seasonal="add",
            seasonal_periods=12
        ).fit(optimized=True)

        # прогноз и метрики
        fc_log = hw_log.forecast(horizon)
        fc = np.expm1(fc_log) 
        # fc   = model.forecast(horizon)
        rmse = np.sqrt(mean_squared_error(test, fc))
        mae  = mean_absolute_error(test, fc)
        mape = (np.abs((test - fc) / test).mean()) * 100

        results_hw.append({
            "Регион":      region,
            "Test start":  test_start.strftime("%Y-%m"),
            "Test end":    test_end.strftime("%Y-%m"),
            "Forecast":    [x.round(2) for x in list(fc.values)],
            "Actual":      [y.round(2) for y in list(test.values)],
            "RMSE":        rmse,
            "MAE":         mae,
            "MAPE_%":      mape
        })

# 4) Усреднение по всем скользящим окнам для каждого региона
res_hw = pd.DataFrame(results_hw)
res_hw.to_excel("results/Верблюды - Результаты прогнозов ХВ v2.xlsx", index=False)

print("Результаты прогнозов HW на 3 месяца:")

display(res_hw)

final_hw = (
    res_hw
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_hw.to_excel("results/Верблюды - Результаты прогнозов ХВ средние v2.xlsx", index=False)
print("Средние метрики Holt–Winter's по регионам (rolling-3):")
display(final_hw)

ГАСТАНА: всего 3 мес. — сезонный Holt-Winter's невозможен.
КОСТАНАЙСКАЯ ОБЛАСТЬ: всего 17 мес. — сезонный Holt-Winter's невозможен.
ОБЛАСТЬ АБАЙ: всего 13 мес. — сезонный Holt-Winter's невозможен.
ОБЛАСТЬ ЖЕТІСУ: всего 20 мес. — сезонный Holt-Winter's невозможен.
ОБЛАСТЬ ҰЛЫТАУ: всего 1 мес. — сезонный Holt-Winter's невозможен.
ПАВЛОДАРСКАЯ ОБЛАСТЬ: всего 22 мес. — сезонный Holt-Winter's невозможен.
СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ: всего 5 мес. — сезонный Holt-Winter's невозможен.
Результаты прогнозов HW на 3 месяца:


,Регион,Test start,Test end,Forecast,Actual,RMSE,MAE,MAPE_%
0,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"[5.8, 50.3, 63.88]","[5.7, 49.87, 63.0]",0.57,0.47,1.34
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"[50.19, 63.75, 156.25]","[49.87, 63.0, 154.3]",1.22,1.01,1.03
2,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"[63.7, 156.12, 550.61]","[63.0, 154.3, 538.4]",7.14,4.91,1.52
3,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"[155.91, 549.86, 122.95]","[154.3, 538.4, 129.3]",7.62,6.47,2.70
4,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"[549.15, 122.79, 73.99]","[538.4, 129.3, 77.6]",7.55,6.96,3.90
...,...,...,...,...,...,...,...,...
77,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"[94.74, 83.49, 127.14]","[69.2, 60.13, 93.1]",28.03,27.65,37.44
78,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"[74.76, 114.06, 103.04]","[60.13, 93.1, 85.8]",17.80,17.61,22.31
79,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"[109.41, 98.89, 105.34]","[93.1, 85.8, 96.61]",13.08,12.71,13.94
80,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"[96.43, 102.75, 105.66]","[85.8, 96.61, 105.0]",7.10,5.81,6.46


Средние метрики Holt–Winter's по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКТЮБИНСКАЯ ОБЛАСТЬ,5.03,4.04,2.23
1,АЛМАТИНСКАЯ ОБЛАСТЬ,12.52,11.06,NaN
2,АТЫРАУСКАЯ ОБЛАСТЬ,107.89,75.17,13.84
3,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2.39,2.00,7.23
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,1.33,1.00,NaN
5,КАРАГАНДИНСКАЯ ОБЛАСТЬ,3.40,2.89,NaN
6,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,323.73,239.23,NaN
7,МАНГИСТАУСКАЯ ОБЛАСТЬ,143.84,115.97,32.34
8,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,85.46,58.47,20.88


## SARIMA

In [11]:
results_sarima = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    ts = ts + epsilon
    ts_log = np.log(ts)

    if len(ts_log) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. для авто-ARIMA, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train_log = ts_log[ts_log.index < test_start]
        test_log  = ts_log[(ts_log.index >= test_start) & (ts_log.index <= test_end)]
        if len(train_log) < 12 + horizon or len(test_log) < horizon:
            continue

        # автоподбор на лог-данных
        use_seasonal = len(train_log) >= 2 * 12

        sarima_log = auto_arima(
            train_log,
            seasonal=use_seasonal,
            m=12 if use_seasonal else 1,
            D=1 if use_seasonal else 0,      # фиксируем порядок сезонной разности
            seasonal_test=None,               # пропустить nsdiffs
            boxcox=True,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore"
        )
      
        # прогноз в лог-шкале
        fc_log = sarima_log.predict(n_periods=horizon, return_conf_int=False)

        # возвращаем прогноз в исходные единицы
        fc = np.exp(fc_log) - epsilon
        actual = np.exp(test_log.values) - epsilon  # но exp(log(x)) == x

        # метрики на исходном уровне
        rmse = np.sqrt(mean_squared_error(actual, fc))
        mae  = mean_absolute_error(actual, fc)
        mape = (np.abs((actual - fc) / actual).mean()) * 100

        results_sarima.append({
            "Регион":         region,
            "Test start":     test_start.strftime("%Y-%m"),
            "Test end":       test_end.strftime("%Y-%m"),
            "order":          sarima_log.order,
            "seasonal_order": sarima_log.seasonal_order,
            "RMSE":           round(rmse,2),
            "MAE":            round(mae,2),
            "MAPE_%":         round(mape,2),
            "Forecast":       [round(x,2) for x in fc],
            "Actual":         [round(y,2) for y in actual]
        })
# формируем DataFrame с результатами
res_sarima = pd.DataFrame(results_sarima)
res_sarima.to_excel("results/Верблюды - Результаты прогнозов SARIMA v2.xlsx", index=False)
print("Результаты прогнозов SARIMA на 3 месяца:")

display(res_sarima)

final_sarima = (
    res_sarima
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_sarima.to_excel("results/Верблюды - Результаты прогнозов SARIMA средние v2.xlsx", index=False)
print("Средние метрики SARIMA по регионам (rolling-3):")
display(final_sarima)

ГАСТАНА: менее 15 мес. для авто-ARIMA, пропускаем.
ОБЛАСТЬ АБАЙ: менее 15 мес. для авто-ARIMA, пропускаем.
ОБЛАСТЬ ҰЛЫТАУ: менее 15 мес. для авто-ARIMA, пропускаем.
СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ: менее 15 мес. для авто-ARIMA, пропускаем.
Результаты прогнозов SARIMA на 3 месяца:


,Регион,Test start,Test end,order,seasonal_order,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"(2, 0, 0)","(0, 1, 0, 12)",0.51,0.36,0.65,"[5.69, 49.66, 62.14]","[5.7, 49.87, 63.0]"
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"(2, 0, 0)","(0, 1, 0, 12)",1.32,1.05,1.03,"[49.68, 62.16, 152.18]","[49.87, 63.0, 154.3]"
2,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"(2, 0, 0)","(0, 1, 0, 12)",2.71,2.31,1.10,"[62.21, 152.33, 534.22]","[63.0, 154.3, 538.4]"
3,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"(2, 0, 0)","(0, 1, 0, 12)",5.66,4.48,2.91,"[152.7, 535.87, 119.97]","[154.3, 538.4, 129.3]"
4,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"(2, 0, 0)","(0, 1, 0, 12)",5.95,5.08,4.46,"[536.95, 120.28, 72.84]","[538.4, 129.3, 77.6]"
...,...,...,...,...,...,...,...,...,...,...
77,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"(3, 0, 0)","(0, 1, 0, 12)",20.83,19.61,25.83,"[82.9, 75.78, 122.57]","[69.2, 60.13, 93.1]"
78,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"(3, 0, 0)","(0, 1, 0, 12)",24.32,23.09,28.62,"[74.49, 115.01, 118.79]","[60.13, 93.1, 85.8]"
79,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"(2, 0, 2)","(0, 1, 0, 12)",37.03,34.84,38.63,"[116.25, 138.04, 125.73]","[93.1, 85.8, 96.61]"
80,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"(3, 0, 0)","(0, 1, 0, 12)",13.17,8.87,10.14,"[108.35, 100.02, 105.64]","[85.8, 96.61, 105.0]"


Средние метрики SARIMA по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКТЮБИНСКАЯ ОБЛАСТЬ,4.62,3.77,2.34
1,АЛМАТИНСКАЯ ОБЛАСТЬ,11.82,10.36,59.56
2,АТЫРАУСКАЯ ОБЛАСТЬ,122.91,88.97,16.87
3,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2.31,1.89,7.06
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,1.34,1.01,7.25
5,КАРАГАНДИНСКАЯ ОБЛАСТЬ,3.78,3.15,49.40
6,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,204.87,152.43,72.67
7,МАНГИСТАУСКАЯ ОБЛАСТЬ,172.36,129.42,43.24
8,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,90.66,62.72,23.49


## Facebook Prophet

In [10]:
results_prophet = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    if len(ts) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. данных, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]
        if len(train) < 12 + horizon or len(test) < horizon:
            continue

        # Подготовка данных для Prophet
#         df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
# # подготовка для одного региона
        df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
        df_prophet["y"] = np.log(df_prophet["y"] + epsilon)

        m = Prophet()
        m.fit(df_prophet)

        future = m.make_future_dataframe(periods=horizon, freq="MS")
        forecast = m.predict(future)

        # берем только прогнозные точки
        yhat_log = forecast["yhat"].values[-horizon:]
        fc = np.exp(yhat_log) - epsilon

        # m = Prophet()
        # m.fit(df_prophet)

        # # Создаем DataFrame будущих дат и делаем прогноз
        # # future = m.make_future_dataframe(periods=horizon, freq="MS")
        # # forecast = m.predict(future)

        # # Отбираем только наши горизонты
        # fc = forecast.set_index("ds")["yhat"].loc[test.index].values
        actual = test.values

        # Расчет метрик
        rmse  = np.sqrt(mean_squared_error(actual, fc))
        mae   = mean_absolute_error(actual, fc)
        mape  = (np.abs((actual - fc) / actual).mean()) * 100

        results_prophet.append({
            "Регион":     region,
            "Test start": test_start.strftime("%Y-%m"),
            "Test end":   test_end.strftime("%Y-%m"),
            "RMSE":       round(rmse, 2),
            "MAE":        round(mae, 2),
            "MAPE_%":     round(mape, 2),
            "Forecast":   [round(x, 2) for x in fc],
            "Actual":     [round(x, 2) for x in actual]
        })

# Собираем результаты в DataFrame
res_prophet = pd.DataFrame(results_prophet)
res_prophet.to_excel("results/Верблюды - Результаты прогнозов Prophet v2.xlsx", index=False)
print("Результаты прогнозов Prophet на 3 месяца:")
display(res_prophet)

final_prophet = (
    res_prophet
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_prophet.to_excel("results/Верблюды - Результаты прогнозов Prophet средние v2.xlsx", index=False)
print("Средние метрики Prophet по регионам (rolling-3):")
display(final_prophet)


18:35:22 - cmdstanpy - INFO - Chain [1] start processing
18:35:24 - cmdstanpy - INFO - Chain [1] done processing
18:35:24 - cmdstanpy - INFO - Chain [1] start processing
18:35:24 - cmdstanpy - INFO - Chain [1] done processing
18:35:25 - cmdstanpy - INFO - Chain [1] start processing
18:35:25 - cmdstanpy - INFO - Chain [1] done processing
18:35:25 - cmdstanpy - INFO - Chain [1] start processing
18:35:25 - cmdstanpy - INFO - Chain [1] done processing
18:35:26 - cmdstanpy - INFO - Chain [1] start processing
18:35:26 - cmdstanpy - INFO - Chain [1] done processing
18:35:26 - cmdstanpy - INFO - Chain [1] start processing
18:35:26 - cmdstanpy - INFO - Chain [1] done processing
18:35:26 - cmdstanpy - INFO - Chain [1] start processing
18:35:26 - cmdstanpy - INFO - Chain [1] done processing
18:35:27 - cmdstanpy - INFO - Chain [1] start processing
18:35:27 - cmdstanpy - INFO - Chain [1] done processing
18:35:27 - cmdstanpy - INFO - Chain [1] start processing
18:35:27 - cmdstanpy - INFO - Chain [1]

ГАСТАНА: менее 15 мес. данных, пропускаем.


18:35:37 - cmdstanpy - INFO - Chain [1] done processing
18:35:37 - cmdstanpy - INFO - Chain [1] start processing
18:35:37 - cmdstanpy - INFO - Chain [1] done processing
18:35:38 - cmdstanpy - INFO - Chain [1] start processing
18:35:38 - cmdstanpy - INFO - Chain [1] done processing
18:35:38 - cmdstanpy - INFO - Chain [1] start processing
18:35:38 - cmdstanpy - INFO - Chain [1] done processing
18:35:38 - cmdstanpy - INFO - Chain [1] start processing
18:35:38 - cmdstanpy - INFO - Chain [1] done processing
18:35:39 - cmdstanpy - INFO - Chain [1] start processing
18:35:39 - cmdstanpy - INFO - Chain [1] done processing
18:35:39 - cmdstanpy - INFO - Chain [1] start processing
18:35:39 - cmdstanpy - INFO - Chain [1] done processing
18:35:40 - cmdstanpy - INFO - Chain [1] start processing
18:35:40 - cmdstanpy - INFO - Chain [1] done processing
18:35:40 - cmdstanpy - INFO - Chain [1] start processing
18:35:40 - cmdstanpy - INFO - Chain [1] done processing
18:35:40 - cmdstanpy - INFO - Chain [1] 

ОБЛАСТЬ АБАЙ: менее 15 мес. данных, пропускаем.
ОБЛАСТЬ ҰЛЫТАУ: менее 15 мес. данных, пропускаем.
СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ: менее 15 мес. данных, пропускаем.


18:35:54 - cmdstanpy - INFO - Chain [1] start processing
18:35:55 - cmdstanpy - INFO - Chain [1] done processing
18:35:56 - cmdstanpy - INFO - Chain [1] start processing
18:35:57 - cmdstanpy - INFO - Chain [1] done processing
18:35:57 - cmdstanpy - INFO - Chain [1] start processing
18:35:58 - cmdstanpy - INFO - Chain [1] done processing
18:35:58 - cmdstanpy - INFO - Chain [1] start processing
18:35:59 - cmdstanpy - INFO - Chain [1] done processing
18:35:59 - cmdstanpy - INFO - Chain [1] start processing
18:36:00 - cmdstanpy - INFO - Chain [1] done processing
18:36:00 - cmdstanpy - INFO - Chain [1] start processing
18:36:01 - cmdstanpy - INFO - Chain [1] done processing
18:36:01 - cmdstanpy - INFO - Chain [1] start processing
18:36:02 - cmdstanpy - INFO - Chain [1] done processing
18:36:03 - cmdstanpy - INFO - Chain [1] start processing
18:36:03 - cmdstanpy - INFO - Chain [1] done processing
18:36:04 - cmdstanpy - INFO - Chain [1] start processing
18:36:05 - cmdstanpy - INFO - Chain [1]

Результаты прогнозов Prophet на 3 месяца:


,Регион,Test start,Test end,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,6.65,4.21,6.90,"[5.69, 48.73, 51.53]","[5.7, 49.87, 63.0]"
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,30.39,21.28,17.85,"[48.79, 51.6, 205.68]","[49.87, 63.0, 154.3]"
2,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,39.30,35.24,20.01,"[51.54, 206.91, 496.74]","[63.0, 154.3, 538.4]"
3,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,38.62,35.64,17.96,"[207.01, 500.63, 112.86]","[154.3, 538.4, 129.3]"
4,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,23.82,21.18,11.11,"[502.2, 112.62, 66.96]","[538.4, 129.3, 77.6]"
...,...,...,...,...,...,...,...,...
77,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,6.86,6.41,8.88,"[72.15, 68.41, 101.11]","[69.2, 60.13, 93.1]"
78,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,7.85,7.84,10.28,"[68.48, 100.99, 93.07]","[60.13, 93.1, 85.8]"
79,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,5.32,4.39,4.90,"[99.94, 91.97, 96.78]","[93.1, 85.8, 96.61]"
80,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,5.73,4.80,4.97,"[91.14, 95.9, 96.65]","[85.8, 96.61, 105.0]"


Средние метрики Prophet по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКТЮБИНСКАЯ ОБЛАСТЬ,22.66,19.19,11.96
1,АЛМАТИНСКАЯ ОБЛАСТЬ,16.24,13.75,95.35
2,АТЫРАУСКАЯ ОБЛАСТЬ,227.59,171.11,28.43
3,ЖАМБЫЛСКАЯ ОБЛАСТЬ,17.19,13.13,39.58
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,3.53,2.89,33.83
5,КАРАГАНДИНСКАЯ ОБЛАСТЬ,4.20,3.77,73.11
6,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,206.07,133.71,46.91
7,МАНГИСТАУСКАЯ ОБЛАСТЬ,146.75,116.67,33.17
8,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,93.73,63.03,17.35


In [12]:
# Переименуем колонки с MAPE, чтобы было понятно, к какому методу относятся
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW"})
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA"})
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet"})

# Мёрджим по региону
summary = (
    hw[["Регион", "MAPE_HW"]]
    .merge(sar[["Регион", "MAPE_SARIMA"]], on="Регион")
    .merge(pr[["Регион", "MAPE_Prophet"]], on="Регион")
)

# Определяем для каждой строки, какой столбец MAPE минимален
# idxmin вернёт название столбца с минимальным значением
summary["Best_method"] = summary[["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]] \
                           .idxmin(axis=1) \
                           .str.replace("MAPE_","")  # убираем префикс для красоты

# Если нужно, можно сразу отсортировать
# summary = summary.sort_values("Best_method")

# допустим, у вас уже есть summary
summary = summary.round({
    "MAPE_HW": 2,
    "MAPE_SARIMA": 2,
    "MAPE_Prophet": 2
})

# Готово!
print(summary.to_string(index=False))
summary.to_excel("results/Верблюды - лучшие модели v2.xlsx", index=False)


                       Регион  MAPE_HW  MAPE_SARIMA  MAPE_Prophet Best_method
          АКТЮБИНСКАЯ ОБЛАСТЬ     2.23         2.34         11.96          HW
          АЛМАТИНСКАЯ ОБЛАСТЬ      NaN        59.56         95.35      SARIMA
           АТЫРАУСКАЯ ОБЛАСТЬ    13.84        16.87         28.43          HW
           ЖАМБЫЛСКАЯ ОБЛАСТЬ     7.23         7.06         39.58      SARIMA
ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ      NaN         7.25         33.83      SARIMA
       КАРАГАНДИНСКАЯ ОБЛАСТЬ      NaN        49.40         73.11      SARIMA
       КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ      NaN        72.67         46.91     Prophet
        МАНГИСТАУСКАЯ ОБЛАСТЬ    32.34        43.24         33.17          HW
        ТУРКЕСТАНСКАЯ ОБЛАСТЬ    20.88        23.49         17.35     Prophet


In [13]:
#FOLDER = Path("results")  # папка, где лежат файлы
FILE_HW      = "results/Верблюды - Результаты прогнозов ХВ средние v2.xlsx"
FILE_SARIMA  = "results/Верблюды - Результаты прогнозов SARIMA средние v2.xlsx"
FILE_PROPHET = "results/Верблюды - Результаты прогнозов Prophet средние v2.xlsx"

OUT_FILE = "Верблюды - Лучшие модели (MAPE_then_MAE) v2.xlsx"
THRESHOLD_MAPE = 50.0  # если минимальный MAPE > 50%, сравниваем по MAE

# === Загрузка исходных таблиц ===
final_hw      = pd.read_excel(FILE_HW)
final_sarima  = pd.read_excel(FILE_SARIMA)
final_prophet = pd.read_excel(FILE_PROPHET)

# Ожидаемые столбцы: 'Регион', 'MAPE_%', 'MAE' (и/или 'RMSE')
# Переименуем для прозрачности
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW", "MAE": "MAE_HW"})[["Регион","MAPE_HW","MAE_HW"]]
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA", "MAE": "MAE_SARIMA"})[["Регион","MAPE_SARIMA","MAE_SARIMA"]]
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet", "MAE": "MAE_Prophet"})[["Регион","MAPE_Prophet","MAE_Prophet"]]

# === Объединяем по региону ===
summary = (
    hw.merge(sar, on="Регион", how="inner")
      .merge(pr,  on="Регион", how="inner")
)


In [14]:
THRESHOLD_MAPE = 1000.0  # порог

mape_cols = ["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]
mae_cols  = ["MAE_HW","MAE_SARIMA","MAE_Prophet"]

# 1) Приведём метрики к числам (на всякий случай ещё раз)
for c in mape_cols + mae_cols:
    summary[c] = pd.to_numeric(summary[c], errors="coerce")

def choose_best_simple(row):
    # Берём числовые серии и подменяем NaN на +inf, чтобы .idxmin() стабильно работал
    mape_s = row[mape_cols].astype(float).fillna(np.inf)
    mae_s  = row[mae_cols].astype(float).fillna(np.inf)

    min_mape = mape_s.min()

    # Если все MAPE были NaN -> min = +inf
    if np.isinf(min_mape):
        criterion = "MAE"
        winner_col = mae_s.idxmin()
    elif min_mape <= THRESHOLD_MAPE:
        criterion = "MAPE"
        winner_col = mape_s.idxmin()
    else:
        criterion = "MAE"
        winner_col = mae_s.idxmin()

    method = winner_col.split("_")[-1]  # HW / SARIMA / Prophet

    return pd.Series({
        "Best_method": method,
        "Best_criterion": criterion,
        "Best_MAPE": float(mape_s.replace(np.inf, np.nan).min()),
        "Best_MAE": float(mae_s.replace(np.inf, np.nan).min())
    })

best = summary.apply(choose_best_simple, axis=1)

result = pd.concat([summary, best], axis=1)

# Округление и сохранение
for c in mape_cols + mae_cols + ["Best_MAPE","Best_MAE"]:
    result[c] = result[c].round(2)

# Если у вас есть переменная OUT_FILE — используйте её. Иначе:
OUT_FILE = "results/Верблюды - Лучшие модели (MAPE_then_MAE) v2.xlsx"
result.sort_values(["Best_method","Регион"]).to_excel(OUT_FILE, index=False)

result

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКТЮБИНСКАЯ ОБЛАСТЬ,2.23,4.04,2.34,3.77,11.96,19.19,HW,MAPE,2.23,3.77
1,АЛМАТИНСКАЯ ОБЛАСТЬ,NaN,11.06,59.56,10.36,95.35,13.75,SARIMA,MAPE,59.56,10.36
2,АТЫРАУСКАЯ ОБЛАСТЬ,13.84,75.17,16.87,88.97,28.43,171.11,HW,MAPE,13.84,75.17
3,ЖАМБЫЛСКАЯ ОБЛАСТЬ,7.23,2.00,7.06,1.89,39.58,13.13,SARIMA,MAPE,7.06,1.89
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,NaN,1.00,7.25,1.01,33.83,2.89,SARIMA,MAPE,7.25,1.00
5,КАРАГАНДИНСКАЯ ОБЛАСТЬ,NaN,2.89,49.40,3.15,73.11,3.77,SARIMA,MAPE,49.40,2.89
6,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,NaN,239.23,72.67,152.43,46.91,133.71,Prophet,MAPE,46.91,133.71
7,МАНГИСТАУСКАЯ ОБЛАСТЬ,32.34,115.97,43.24,129.42,33.17,116.67,HW,MAPE,32.34,115.97
8,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,20.88,58.47,23.49,62.72,17.35,63.03,Prophet,MAPE,17.35,58.47
